In [20]:
!pip install -q "transformers>=4.40.0" accelerate torchvision pycocotools roboflow

import torch
assert torch.cuda.is_available(), "Switch Colab runtime to T4 GPU: Runtime -> Change runtime type -> T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU: Tesla T4 | VRAM: 15.64 GB


Cell 2: Dataset & $C$-Way $K$-Shot Sampler (NEU-DET / COCO Format)

In [21]:
import os
import json
import random
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_convert

def create_demo_neudet_coco(root_dir="/content/neudet_demo", num_images=24):
    """Creates a realistic synthetic 6-class NEU-DET COCO structure so the notebook runs out-of-the-box."""
    os.makedirs(f"{root_dir}/images", exist_ok=True)
    classes = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
    categories = [{"id": i, "name": name} for i, name in enumerate(classes)]
    images, annotations = [], []
    ann_id = 1

    for img_id in range(1, num_images + 1):
        W, H = 400, 400
        # Textured grayscale-like industrial background with simulated defect patches
        arr = np.random.normal(128, 25, (H, W, 3)).clip(0, 255).astype(np.uint8)
        cls_id = img_id % len(classes)
        x1, y1 = random.randint(40, 200), random.randint(40, 200)
        w, h = random.randint(60, 140), random.randint(60, 140)
        arr[y1:y1+h, x1:x1+w] = np.clip(arr[y1:y1+h, x1:x1+w] * 0.4, 0, 255)

        file_name = f"img_{img_id:03d}.jpg"
        Image.fromarray(arr).save(f"{root_dir}/images/{file_name}")
        images.append({"id": img_id, "file_name": file_name, "width": W, "height": H})
        annotations.append({
            "id": ann_id, "image_id": img_id, "category_id": cls_id,
            "bbox": [float(x1), float(y1), float(w), float(h)], "area": float(w * h), "iscrowd": 0
        })
        ann_id += 1

    coco_dict = {"images": images, "annotations": annotations, "categories": categories}
    ann_path = f"{root_dir}/annotations.json"
    with open(ann_path, "w") as f:
        json.dump(coco_dict, f)
    return f"{root_dir}/images", ann_path, classes


class CDFSODDataset(Dataset):
    """Loads a COCO-formatted dataset and splits into K-shot Support Set or Query Set."""
    def __init__(self, img_dir, ann_List, processor, class_names, max_size=800):
        self.img_dir = img_dir
        self.samples = ann_List
        self.processor = processor
        self.class_names = class_names
        self.max_size = max_size
        # GroundingDINO text prompt format: "crazing . inclusion . patches ."
        self.text_prompt = " . ".join(class_names) + " ."

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        img_path = os.path.join(self.img_dir, item["file_name"])
        image = Image.open(img_path).convert("RGB")
        W, H = image.size

        boxes_xywh = torch.tensor(item["bboxes"], dtype=torch.float32)  # [J, 4] (x, y, w, h)
        labels = torch.tensor(item["category_ids"], dtype=torch.long)   # [J]

        # Convert to xyxy (absolute pixels) for RoIAlign & Hard Negative Jittering
        boxes_xyxy = box_convert(boxes_xywh, in_fmt="xywh", out_fmt="xyxy")
        # Convert to normalized cxcywh [0, 1] for DETR/GroundingDINO SetCriterion
        boxes_cxcywh_norm = box_convert(boxes_xywh, in_fmt="xywh", out_fmt="cxcywh")
        boxes_cxcywh_norm[:, [0, 2]] /= W
        boxes_cxcywh_norm[:, [1, 3]] /= H

        encoding = self.processor(
            images=image,
            text=self.text_prompt,
            size={"shortest_edge": 480, "longest_edge": self.max_size},
            return_tensors="pt",
        )
        # Squeeze batch dimension added by processor
        pixel_values = encoding["pixel_values"].squeeze(0)
        pixel_mask = encoding["pixel_mask"].squeeze(0)
        _, H_proc, W_proc = pixel_values.shape

        # Scale xyxy boxes to match the resized tensor dimensions (H_proc, W_proc)
        scale_x, scale_y = W_proc / float(W), H_proc / float(H)
        boxes_xyxy_proc = boxes_xyxy.clone()
        boxes_xyxy_proc[:, [0, 2]] *= scale_x
        boxes_xyxy_proc[:, [1, 3]] *= scale_y

        return {
            "pixel_values": pixel_values,
            "pixel_mask": pixel_mask,
            "boxes_xyxy": boxes_xyxy_proc,
            "boxes_cxcywh_norm": boxes_cxcywh_norm,
            "labels": labels,
            "img_shape": (H_proc, W_proc),
        }


def build_k_shot_splits(ann_path, k_shot=5, seed=42):
    """Samples K annotated instances per class C for the support set, remaining for query."""
    random.seed(seed)
    with open(ann_path, "r") as f:
        coco = json.load(f)

    cat_id_to_idx = {c["id"]: idx for idx, c in enumerate(coco["categories"])}
    class_names = [c["name"] for c in coco["categories"]]

    img_to_anns = {}
    for ann in coco["annotations"]:
        img_to_anns.setdefault(ann["image_id"], []).append(ann)

    all_records = []
    for img in coco["images"]:
        anns = img_to_anns.get(img["id"], [])
        if not anns:
            continue
        all_records.append({
            "image_id": img["id"],
            "file_name": img["file_name"],
            "bboxes": [a["bbox"] for a in anns],
            "category_ids": [cat_id_to_idx[a["category_id"]] for a in anns],
        })

    random.shuffle(all_records)
    class_counts = {c: 0 for c in range(len(class_names))}
    support_records, query_records = [], []

    for rec in all_records:
        needed = any(class_counts[c] < k_shot for c in rec["category_ids"])
        if needed:
            support_records.append(rec)
            for c in rec["category_ids"]:
                class_counts[c] += 1
        else:
            query_records.append(rec)

    if not query_records:
        query_records = support_records.copy()

    return support_records, query_records, class_names


def collate_fn(batch):
    # Pad pixel_values to max height/width in batch
    max_h = max(b["pixel_values"].shape[1] for b in batch)
    max_w = max(b["pixel_values"].shape[2] for b in batch)
    B = len(batch)
    padded_imgs = torch.zeros((B, 3, max_h, max_w), dtype=torch.float32)
    padded_masks = torch.zeros((B, max_h, max_w), dtype=torch.bool)

    for i, b in enumerate(batch):
        _, h, w = b["pixel_values"].shape
        padded_imgs[i, :, :h, :w] = b["pixel_values"]
        padded_masks[i, :h, :w] = b["pixel_mask"].bool()

    return {
        "pixel_values": padded_imgs,
        "pixel_mask": padded_masks,
        "boxes_xyxy": [b["boxes_xyxy"] for b in batch],
        "targets": [
            {"boxes": b["boxes_cxcywh_norm"], "labels": b["labels"]}
            for b in batch
        ],
        "labels": [b["labels"] for b in batch],
        "img_shapes": [b["img_shape"] for b in batch],
    }

Cell 3: Visual Prototype Construction & Hard Negative Jittering (Section 3.3)

In [22]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import roi_align, box_iou

class VisualPrototypeConstructor(nn.Module):
    def __init__(
        self,
        num_classes: int,
        hidden_dim: int = 256,
        roi_size: int = 7,
        num_negatives_per_gt: int = 3,
        scale_range: tuple = (0.6, 1.0),
        shift_ratio: float = 0.2,
        iou_range: tuple = (0.1, 0.5),
    ):
        super().__init__()
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.roi_size = roi_size
        self.num_negatives = num_negatives_per_gt
        self.scale_range = scale_range
        self.shift_ratio = shift_ratio
        self.iou_range = iou_range

        self.register_buffer("cached_class_prototypes", torch.zeros(num_classes, hidden_dim))

    def _extract_roi_descriptors(self, feat_map, boxes_xyxy_list, img_shapes):
        """Applies RoIAlign + GAP on Level-0 feature map f^{(0)} and L2-normalizes."""
        B, C, H_f, W_f = feat_map.shape
        descriptors = []

        for b_idx in range(B):
            boxes = boxes_xyxy_list[b_idx]
            if boxes.numel() == 0:
                continue
            H_img, W_img = img_shapes[b_idx]
            scale_x, scale_y = W_f / float(W_img), H_f / float(H_img)
            scaled_boxes = boxes.clone()
            scaled_boxes[:, [0, 2]] *= scale_x
            scaled_boxes[:, [1, 3]] *= scale_y

            rois = roi_align(
                feat_map[b_idx : b_idx + 1].float(),
                [scaled_boxes.float()],
                output_size=(self.roi_size, self.roi_size),
                spatial_scale=1.0,
                aligned=True,
            )
            gap = rois.mean(dim=[-2, -1])  # [M_b, D_I]
            descriptors.append(gap)

        if not descriptors:
            return torch.empty((0, C), device=feat_map.device, dtype=torch.float32)
        return F.normalize(torch.cat(descriptors, dim=0), p=2, dim=-1)

    @torch.no_grad()
    def sample_hard_negative_boxes(self, gt_boxes_xyxy: torch.Tensor, img_shape: tuple) -> torch.Tensor:
        """Samples N=3 jittered boxes per GT box with IoU in [0.1, 0.5] (Eq. 1)."""
        if gt_boxes_xyxy.numel() == 0 or self.num_negatives == 0:
            return torch.empty((0, 4), device=gt_boxes_xyxy.device)

        H_img, W_img = img_shape
        device = gt_boxes_xyxy.device
        gt_cxcywh = box_convert(gt_boxes_xyxy, in_fmt="xyxy", out_fmt="cxcywh")
        M = 120  # Parallel rejection sampling candidates
        collected = []

        for j in range(gt_cxcywh.shape[0]):
            cx, cy, w, h = gt_cxcywh[j]
            box_j = gt_boxes_xyxy[j : j + 1]

            s_w = torch.empty(M, device=device).uniform_(*self.scale_range)
            s_h = torch.empty(M, device=device).uniform_(*self.scale_range)
            dx = torch.empty(M, device=device).uniform_(-self.shift_ratio * w, self.shift_ratio * w)
            dy = torch.empty(M, device=device).uniform_(-self.shift_ratio * h, self.shift_ratio * h)

            cand_cxcywh = torch.stack([cx + dx, cy + dy, w * s_w, h * s_h], dim=-1)
            cand_xyxy = box_convert(cand_cxcywh, in_fmt="cxcywh", out_fmt="xyxy")
            cand_xyxy[:, [0, 2]] = cand_xyxy[:, [0, 2]].clamp(0, W_img)
            cand_xyxy[:, [1, 3]] = cand_xyxy[:, [1, 3]].clamp(0, H_img)

            ious = box_iou(cand_xyxy, box_j).squeeze(-1)
            valid = cand_xyxy[(ious >= self.iou_range[0]) & (ious <= self.iou_range[1])]

            # Fallback if strict 0.2 shift produces IoU > 0.5: widen center shift to hit [0.1, 0.5] boundary
            if valid.shape[0] < self.num_negatives:
                dx_w = torch.empty(M, device=device).uniform_(-0.65 * w, 0.65 * w)
                dy_w = torch.empty(M, device=device).uniform_(-0.65 * h, 0.65 * h)
                cand_w = box_convert(torch.stack([cx + dx_w, cy + dy_w, w * s_w, h * s_h], dim=-1), "cxcywh", "xyxy")
                cand_w[:, [0, 2]] = cand_w[:, [0, 2]].clamp(0, W_img)
                cand_w[:, [1, 3]] = cand_w[:, [1, 3]].clamp(0, H_img)
                ious_w = box_iou(cand_w, box_j).squeeze(-1)
                valid = torch.cat([valid, cand_w[(ious_w >= self.iou_range[0]) & (ious_w <= self.iou_range[1])]], dim=0)

            if valid.shape[0] >= self.num_negatives:
                collected.append(valid[: self.num_negatives])
            elif valid.shape[0] > 0:
                reps = (self.num_negatives + valid.shape[0] - 1) // valid.shape[0]
                collected.append(valid.repeat(reps, 1)[: self.num_negatives])

        return torch.cat(collected, dim=0) if collected else torch.empty((0, 4), device=device)

    def update_class_prototypes(self, support_feats_list, support_boxes_list, support_labels_list, support_shapes_list):
        """Aggregates class prototypes P_cls across all support instances."""
        all_descs = []
        for feat, boxes, shapes in zip(support_feats_list, support_boxes_list, support_shapes_list):
            desc = self._extract_roi_descriptors(feat, boxes, shapes)
            all_descs.append(desc)
        all_descs = torch.cat(all_descs, dim=0)
        all_labels = torch.cat(support_labels_list, dim=0)

        P_cls = torch.zeros((self.num_classes, self.hidden_dim), device=all_descs.device, dtype=torch.float32)
        for c in range(self.num_classes):
            mask = all_labels == c
            if mask.any():
                P_cls[c] = F.normalize(all_descs[mask].mean(dim=0), p=2, dim=-1)

        self.cached_class_prototypes.copy_(P_cls.detach())
        return P_cls

    def forward(self, query_level0_feat, query_gt_boxes_xyxy=None, query_img_shapes=None):
        """Constructs V = [P_cls; P_neg] \in R^{N_V x D_I} and attention masks."""
        B, D_I, _, _ = query_level0_feat.shape
        device = query_level0_feat.device
        P_cls = self.cached_class_prototypes

        if not self.training or query_gt_boxes_xyxy is None:
            V_batch = P_cls.unsqueeze(0).expand(B, -1, -1).contiguous()
            V_mask = torch.ones((B, self.num_classes), dtype=torch.bool, device=device)
            return V_batch, V_mask

        V_list, max_len = [], self.num_classes
        for b_idx in range(B):
            neg_boxes = self.sample_hard_negative_boxes(query_gt_boxes_xyxy[b_idx], query_img_shapes[b_idx])
            if neg_boxes.numel() > 0:
                P_neg = self._extract_roi_descriptors(
                    query_level0_feat[b_idx : b_idx + 1], [neg_boxes], [query_img_shapes[b_idx]]
                )
                V_b = torch.cat([P_cls, P_neg], dim=0)
            else:
                V_b = P_cls
            V_list.append(V_b)
            max_len = max(max_len, V_b.shape[0])

        V_batch = torch.zeros((B, max_len, D_I), device=device, dtype=torch.float32)
        V_mask = torch.zeros((B, max_len), dtype=torch.bool, device=device)
        for b_idx, V_b in enumerate(V_list):
            L = V_b.shape[0]
            V_batch[b_idx, :L] = V_b
            V_mask[b_idx, :L] = True

        return V_batch, V_mask

<>:122: SyntaxWarning: invalid escape sequence '\i'
<>:122: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipykernel_442/1497220278.py:122: SyntaxWarning: invalid escape sequence '\i'
  """Constructs V = [P_cls; P_neg] \in R^{N_V x D_I} and attention masks."""


#Cell 4: Dual-Branch LMP Architecture Using Hugging Face GroundingDINO (Sections 3.4 & 3.5)
In Hugging Face's GroundingDinoModel, the modules map 1-to-1 to the paper:model.backbone + model.input_proj: Shared Swin-B + 4-level feature pyramid ($1/8, 1/16, 1/32, 1/64$).   model.encoder: 6-layer Feature Enhancer (Self-Attention + Bi-directional Image$\leftrightarrow$Prototype Cross-Attention).   model.enc_output + model.enc_output_norm: Guided Query Selection ($N_q = 900$ Top-K cosine similarity).   model.decoder: 6-layer Cross-Modality / Visual Decoder.   We deep-copy encoder, enc_output, enc_output_norm, decoder, and bbox_embed to form the Visual-Guided Branch initialized from the Text-Guided Branch.   

In [23]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers.models.grounding_dino.modeling_grounding_dino import GroundingDinoSinePositionEmbedding

class CosinePrototypeHead(nn.Module):
    """Section 3.6: z_{i,c} = (\alpha / \tau) * cos(W_cls * q_i, p_c) over C classes only."""
    def __init__(self, hidden_dim=256, init_alpha=1.0, tau=0.07):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.alpha = nn.Parameter(torch.tensor(init_alpha))
        self.tau = tau

    def forward(self, query_embeds, class_prototypes):
        q_norm = F.normalize(self.proj(query_embeds), p=2, dim=-1)
        p_norm = F.normalize(class_prototypes, p=2, dim=-1)
        return (self.alpha / self.tau) * torch.einsum("bqd,bcd->bqc", q_norm, p_norm)


class HF_LMPGroundingDINO(nn.Module):
    def __init__(self, pretrained_ckpt: str, num_classes: int, num_negatives: int = 3):
        super().__init__()
        base = AutoModelForZeroShotObjectDetection.from_pretrained(pretrained_ckpt)
        gdino = base.model
        self.config = base.config
        self.num_classes = num_classes
        self.hidden_dim = gdino.config.d_model  # 256
        self.num_queries = gdino.config.num_queries  # 900

        # 1. Shared Image Backbone & Multi-scale Pyramid Projection
        self.backbone = gdino.backbone

        # Actual channels verified from backbone diagnostic
        backbone_channels = [256, 512, 1024]

        # Three backbone levels + one additional pyramid level
        self.input_proj = nn.ModuleList([
            # Level 0: 256 -> 256
            nn.Sequential(
                nn.Conv2d(256, self.hidden_dim, kernel_size=1),
                nn.GroupNorm(32, self.hidden_dim)
            ),

            # Level 1: 512 -> 256
            nn.Sequential(
                nn.Conv2d(512, self.hidden_dim, kernel_size=1),
                nn.GroupNorm(32, self.hidden_dim)
            ),

            # Level 2: 1024 -> 256
            nn.Sequential(
                nn.Conv2d(1024, self.hidden_dim, kernel_size=1),
                nn.GroupNorm(32, self.hidden_dim)
            ),

            # Level 3: Additional 1/64 feature level
            nn.Sequential(
                nn.Conv2d(
                    1024,
                    self.hidden_dim,
                    kernel_size=3,
                    stride=2,
                    padding=1
                ),
                nn.GroupNorm(32, self.hidden_dim)
            )
        ])

        self.position_embedding = GroundingDinoSinePositionEmbedding(
            gdino.config
        )
        self.level_embed = gdino.level_embed if hasattr(gdino, "level_embed") else nn.Parameter(torch.zeros(4, self.hidden_dim))

        # Freeze BERT text encoder to save ~1.3 GB VRAM on Colab T4
        self.text_backbone = gdino.text_backbone
        for p in self.text_backbone.parameters():
            p.requires_grad = False
        self.text_projection = gdino.text_projection  # Trainable W_t (768 -> 256)

        # 2. Text Branch Components
        self.text_encoder = gdino.encoder
        self.text_enc_output = gdino.enc_output
        self.text_enc_output_norm = gdino.enc_output_norm
        self.text_encoder_vision_head = gdino.encoder_output_bbox_embed
        self.text_query_pos = gdino.query_position_embeddings
        self.text_decoder = gdino.decoder
        self.text_bbox_embed = base.bbox_embed
        self.text_cls_head = CosinePrototypeHead(self.hidden_dim)

        # 3. Visual Branch (Mirrored & initialized from Text Branch per Section 3.4)
        self.vis_encoder = copy.deepcopy(gdino.encoder)
        self.vis_enc_output = copy.deepcopy(gdino.enc_output)
        self.vis_enc_output_norm = copy.deepcopy(gdino.enc_output_norm)
        self.vis_encoder_vision_head = copy.deepcopy(gdino.encoder_output_bbox_embed)
        self.vis_query_pos = copy.deepcopy(gdino.query_position_embeddings)
        self.vis_decoder = copy.deepcopy(gdino.decoder)
        self.vis_bbox_embed = copy.deepcopy(base.bbox_embed)
        self.vis_cls_head = copy.deepcopy(self.text_cls_head)

        # 4. Visual Prototype Constructor
        self.prototype_constructor = VisualPrototypeConstructor(
            num_classes=num_classes, hidden_dim=self.hidden_dim, num_negatives_per_gt=num_negatives
        )

    def extract_multi_scale_features(self, pixel_values, pixel_mask):
        """Extracts a four-level multi-scale feature pyramid."""

        features, pos_list = self.backbone(pixel_values, pixel_mask)

        srcs, masks, pos_embeds = [], [], []

        # First three backbone levels
        for lvl, (feat_tensor, mask_tensor) in enumerate(features):

            src = self.input_proj[lvl](feat_tensor)

            srcs.append(src)
            masks.append(mask_tensor)
            pos_embeds.append(pos_list[lvl])

        # Fourth level: downsample the last raw backbone feature
        last_feat, last_mask = features[-1]

        src_l3 = self.input_proj[3](last_feat)

        mask_l3 = F.interpolate(
            last_mask[:, None].float(),
            size=src_l3.shape[-2:],
            mode="nearest"
        )[:, 0].bool()

        pos_l3 = self.position_embedding(src_l3, mask_l3)

        srcs.append(src_l3)
        masks.append(mask_l3)
        pos_embeds.append(pos_l3)

        return srcs, masks, pos_embeds

    def encode_class_text_prototypes(self, class_names, tokenizer, device):
        """Encodes C class names into P_text \in R^{C x 256} (Section 3.5)."""
        tok = tokenizer(class_names, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            bert_out = self.text_backbone(
                input_ids=tok["input_ids"], attention_mask=tok["attention_mask"]
            ).last_hidden_state
        mask = tok["attention_mask"].unsqueeze(-1).float()
        pooled = (bert_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        P_text = F.normalize(self.text_projection(pooled.to(self.text_projection.weight.dtype)), p=2, dim=-1)
        return P_text

    def _run_single_branch(
        self, srcs, masks, pos_embeds, proto_tokens, proto_mask,
        encoder, enc_output, enc_output_norm, enc_bbox_head, query_pos_embed, decoder, bbox_embed, cls_head
    ):
        """Executes Feature Enhancer -> Top-900 Query Selection -> 6-Layer Decoder."""
        B = srcs[0].shape[0]
        device = srcs[0].device

        # Flatten multi-scale image features
        feat_flatten, mask_flatten, lvl_pos_flatten, spatial_shapes = [], [], [], []
        for lvl, (src, mask, pos) in enumerate(zip(srcs, masks, pos_embeds)):
            _, _, h, w = src.shape
            spatial_shapes.append((h, w))
            feat_flatten.append(src.flatten(2).transpose(1, 2))
            mask_flatten.append(mask.flatten(1))
            lvl_pos_flatten.append(pos.flatten(2).transpose(1, 2) + self.level_embed[lvl].view(1, 1, -1))

        feat_flatten = torch.cat(feat_flatten, dim=1)          # [B, N_I, 256]
        mask_flatten = torch.cat(mask_flatten, dim=1)          # [B, N_I] (True = valid in HF)
        lvl_pos_flatten = torch.cat(lvl_pos_flatten, dim=1)    # [B, N_I, 256]
        spatial_shapes_t = torch.as_tensor(spatial_shapes, dtype=torch.long, device=device)
        level_start_index = torch.cat((spatial_shapes_t.new_zeros((1,)), spatial_shapes_t.prod(1).cumsum(0)[:-1]))
        valid_ratios = torch.ones((B, len(srcs), 2), device=device, dtype=feat_flatten.dtype)

        # Self-attention mask for prototypes
        N_P = proto_tokens.shape[1]
        proto_self_mask = torch.eye(N_P, device=device, dtype=torch.bool).unsqueeze(0).expand(B, -1, -1)
        proto_pos_ids = torch.arange(N_P, device=device).unsqueeze(0).expand(B, -1)

        # 1. Feature Enhancer (6 Bi-directional Cross-Attention Layers)
        enc_outputs = encoder(
            vision_features=feat_flatten,
            vision_attention_mask=mask_flatten,
            vision_position_embedding=lvl_pos_flatten,
            spatial_shapes=spatial_shapes_t,
            spatial_shapes_list=spatial_shapes,
            level_start_index=level_start_index,
            valid_ratios=valid_ratios,
            text_features=proto_tokens.to(feat_flatten.dtype),
            text_attention_mask=proto_mask,
            text_position_embedding=None,
            text_self_attention_masks=proto_self_mask,
            text_position_ids=proto_pos_ids,
        )
        X_I_prime = enc_outputs.last_hidden_state_vision  # [B, N_I, 256]
        V_prime = enc_outputs.last_hidden_state_text      # [B, N_P, 256]

        # 2. Guided Query Selection
        proposals_list = []
        for lvl, (h, w) in enumerate(spatial_shapes):
            grid_y, grid_x = torch.meshgrid(
                torch.linspace(0.5, h - 0.5, h, device=device) / h,
                torch.linspace(0.5, w - 0.5, w, device=device) / w,
                indexing="ij",
            )
            scale = torch.full_like(grid_x, 0.05 * (2.0 ** lvl))
            prop = torch.stack([grid_x, grid_y, scale, scale], dim=-1).view(1, -1, 4).expand(B, -1, -1)
            proposals_list.append(prop)
        proposals = torch.cat(proposals_list, dim=1).clamp(0.01, 0.99)
        proposals_logit = torch.logit(proposals)

        enc_mem = enc_output_norm(enc_output(X_I_prime))
        sim_matrix = torch.einsum(
            "bid,bpd->bip",
            F.normalize(enc_mem, p=2, dim=-1),
            F.normalize(V_prime, p=2, dim=-1),
        )
        sim_matrix = sim_matrix.masked_fill(~proto_mask.unsqueeze(1), float("-inf"))
        max_sim = sim_matrix.max(dim=-1).values
        topk_idx = torch.topk(max_sim, k=self.num_queries, dim=1).indices

        topk_idx_4 = topk_idx.unsqueeze(-1).expand(-1, -1, 4)
        init_ref_logit = torch.gather(proposals_logit, 1, topk_idx_4) + torch.gather(enc_bbox_head(enc_mem), 1, topk_idx_4)
        init_ref_points = init_ref_logit.sigmoid().detach()

        target_queries = query_pos_embed.weight.unsqueeze(0).expand(B, -1, -1)

        # 3. Prototype-Aware Decoder (6 Layers)
        dec_outputs = decoder(
            inputs_embeds=target_queries.to(X_I_prime.dtype),
            vision_encoder_hidden_states=X_I_prime,
            vision_encoder_attention_mask=mask_flatten,
            text_encoder_hidden_states=V_prime,
            text_encoder_attention_mask=proto_mask,
            reference_points=init_ref_points,
            spatial_shapes=spatial_shapes_t,
            spatial_shapes_list=spatial_shapes,
            level_start_index=level_start_index,
            valid_ratios=valid_ratios,
            self_attn_mask=None,
            output_hidden_states=True,
        )

        last_hs = dec_outputs.last_hidden_state
        last_ref = init_ref_points
        delta_box = bbox_embed[-1](last_hs) if isinstance(bbox_embed, nn.ModuleList) else bbox_embed(last_hs)
        pred_boxes = (delta_box + torch.logit(last_ref.clamp(1e-5, 1 - 1e-5))).sigmoid()

        adapted_class_protos = V_prime[:, : self.num_classes, :]
        pred_logits = cls_head(last_hs, adapted_class_protos)

        return {"pred_logits": pred_logits, "pred_boxes": pred_boxes}

    def forward(self, pixel_values, pixel_mask, class_names, tokenizer, gt_boxes_xyxy=None, img_shapes=None, stage=2):
        srcs, masks, pos_embeds = self.extract_multi_scale_features(pixel_values, pixel_mask)

        V_batch, V_mask = self.prototype_constructor(
            query_level0_feat=srcs[0],
            query_gt_boxes_xyxy=gt_boxes_xyxy,
            query_img_shapes=img_shapes,
        )

        visual_out = self._run_single_branch(
            srcs, masks, pos_embeds, V_batch, V_mask,
            self.vis_encoder, self.vis_enc_output, self.vis_enc_output_norm,
            self.vis_encoder_vision_head, self.vis_query_pos, self.vis_decoder,
            self.vis_bbox_embed, self.vis_cls_head
        )
        if stage == 1:
            return {"visual": visual_out}

        B = srcs[0].shape[0]
        P_text = self.encode_class_text_prototypes(class_names, tokenizer, srcs[0].device)
        P_text_batch = P_text.unsqueeze(0).expand(B, -1, -1).contiguous()
        P_text_mask = torch.ones((B, self.num_classes), dtype=torch.bool, device=srcs[0].device)

        text_out = self._run_single_branch(
            srcs, masks, pos_embeds, P_text_batch, P_text_mask,
            self.text_encoder, self.text_enc_output, self.text_enc_output_norm,
            self.text_encoder_vision_head, self.text_query_pos, self.text_decoder,
            self.text_bbox_embed, self.text_cls_head
        )
        return {"visual": visual_out, "text": text_out}

<>:143: SyntaxWarning: invalid escape sequence '\i'
<>:143: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipykernel_442/4050296592.py:143: SyntaxWarning: invalid escape sequence '\i'
  """Encodes C class names into P_text \in R^{C x 256} (Section 3.5)."""


#Cell 5: Hungarian Matcher & Focal + L1 + GIoU Loss (Supplementary Section A)
Implements the exact matching costs (1.0, 5.0, 2.0) and loss weights (2.0, 5.0, 2.0) specified in Supplementary Section A.   

In [24]:
from scipy.optimize import linear_sum_assignment
from torchvision.ops import generalized_box_iou, sigmoid_focal_loss

class LMPSetCriterion(nn.Module):
    def __init__(self, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0, weight_cls=2.0, weight_bbox=5.0, weight_giou=2.0):
        super().__init__()
        self.cost_class = cost_class
        self.cost_bbox = cost_bbox
        self.cost_giou = cost_giou
        self.weight_cls = weight_cls
        self.weight_bbox = weight_bbox
        self.weight_giou = weight_giou

    @torch.no_grad()
    def match(self, pred_logits, pred_boxes, targets):
        B, N_q, C = pred_logits.shape
        indices = []
        for b in range(B):
            tgt_ids = targets[b]["labels"]
            tgt_bbox = targets[b]["boxes"]
            if tgt_ids.numel() == 0:
                indices.append((torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)))
                continue

            out_prob = pred_logits[b].float().sigmoid()  # [900, C]
            out_bbox = pred_boxes[b].float()             # [900, 4]

            # Focal classification cost
            alpha, gamma = 0.25, 2.0
            neg_cost = (1 - alpha) * (out_prob ** gamma) * (-(1 - out_prob + 1e-8).log())
            pos_cost = alpha * ((1 - out_prob) ** gamma) * (-(out_prob + 1e-8).log())
            cost_class = pos_cost[:, tgt_ids] - neg_cost[:, tgt_ids]

            # L1 and GIoU box costs
            cost_bbox = torch.cdist(out_bbox, tgt_bbox, p=1)
            cost_giou = -generalized_box_iou(
                box_convert(out_bbox, "cxcywh", "xyxy"),
                box_convert(tgt_bbox, "cxcywh", "xyxy"),
            )

            C_mat = self.cost_class * cost_class + self.cost_bbox * cost_bbox + self.cost_giou * cost_giou
            row_ind, col_ind = linear_sum_assignment(C_mat.cpu().numpy())
            indices.append((
                torch.as_tensor(row_ind, dtype=torch.long, device=pred_logits.device),
                torch.as_tensor(col_ind, dtype=torch.long, device=pred_logits.device),
            ))
        return indices

    def forward(self, outputs, targets):
        pred_logits = outputs["pred_logits"].float()
        pred_boxes = outputs["pred_boxes"].float()
        indices = self.match(pred_logits, pred_boxes, targets)

        B, N_q, C = pred_logits.shape
        device = pred_logits.device
        num_boxes = max(sum(len(t["labels"]) for t in targets), 1)

        # 1. One-hot Focal Target (hard negatives have no class column, so distractors get 0 target everywhere)
        target_classes_onehot = torch.zeros((B, N_q, C), dtype=torch.float32, device=device)
        loss_bbox_total, loss_giou_total = 0.0, 0.0

        for b, (src_idx, tgt_idx) in enumerate(indices):
            if src_idx.numel() == 0:
                continue
            tgt_labels = targets[b]["labels"][tgt_idx]
            target_classes_onehot[b, src_idx, tgt_labels] = 1.0

            src_boxes = pred_boxes[b, src_idx]
            tgt_boxes = targets[b]["boxes"][tgt_idx]
            loss_bbox_total += F.l1_loss(src_boxes, tgt_boxes, reduction="sum")
            giou = torch.diag(generalized_box_iou(
                box_convert(src_boxes, "cxcywh", "xyxy"),
                box_convert(tgt_boxes, "cxcywh", "xyxy"),
            ))
            loss_giou_total += (1.0 - giou).sum()

        loss_cls = sigmoid_focal_loss(pred_logits, target_classes_onehot, alpha=0.25, gamma=2.0, reduction="sum") / num_boxes
        loss_bbox = loss_bbox_total / num_boxes
        loss_giou = loss_giou_total / num_boxes

        return self.weight_cls * loss_cls + self.weight_bbox * loss_bbox + self.weight_giou * loss_giou

# Importing Database

In [25]:
# Option 2A: Pull NEU-DET directly via Kaggle CLI in Colab (No API key needed for public mirror)
!pip install -q kagglehub pycocotools
import kagglehub, os, glob, xml.etree.ElementTree as ET, json

path = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")
print("Downloaded NEU-DET to:", path)

def convert_voc_to_coco(voc_root, out_json="/content/neudet_real_coco.json"):
    """Converts NEU-DET Pascal VOC XML annotations into a single COCO JSON file."""
    classes = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
    cls_to_id = {name: i for i, name in enumerate(classes)}
    categories = [{"id": i, "name": name} for i, name in enumerate(classes)]
    images, annotations = [], []
    ann_id = 1

    xml_files = sorted(glob.glob(os.path.join(voc_root, "**", "*.xml"), recursive=True))
    img_files = {os.path.basename(p): p for p in glob.glob(os.path.join(voc_root, "**", "*.jpg"), recursive=True)}
    img_files.update({os.path.basename(p): p for p in glob.glob(os.path.join(voc_root, "**", "*.bmp"), recursive=True)})

    for img_id, xml_path in enumerate(xml_files, 1):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        stem = os.path.splitext(os.path.basename(xml_path))[0]
        # Match either .jpg or .bmp
        fname = next((k for k in [f"{stem}.jpg", f"{stem}.bmp"] if k in img_files), None)
        if not fname:
            continue

        size = root.find("size")
        W, H = int(size.find("width").text), int(size.find("height").text)
        images.append({"id": img_id, "file_name": img_files[fname], "width": W, "height": H})

        for obj in root.findall("object"):
            cname = obj.find("name").text.strip()
            if cname not in cls_to_id:
                continue
            bnd = obj.find("bndbox")
            x1, y1 = float(bnd.find("xmin").text), float(bnd.find("ymin").text)
            x2, y2 = float(bnd.find("xmax").text), float(bnd.find("ymax").text)
            w, h = max(1.0, x2 - x1), max(1.0, y2 - y1)
            annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": cls_to_id[cname],
                "bbox": [x1, y1, w, h], "area": w * h, "iscrowd": 0
            })
            ann_id += 1

    with open(out_json, "w") as f:
        json.dump({"images": images, "annotations": annotations, "categories": categories}, f)
    print(f"Converted {len(images)} real NEU-DET images with {len(annotations)} defect boxes -> {out_json}")
    return "", out_json, classes

# Run conversion and plug directly into Cell 6:
img_dir, ann_path, class_names = convert_voc_to_coco(path)

Using Colab cache for faster access to the 'neu-surface-defect-database' dataset.
Downloaded NEU-DET to: /kaggle/input/neu-surface-defect-database
Converted 1800 real NEU-DET images with 4189 defect boxes -> /content/neudet_real_coco.json


#Cell 6: Two-Stage Training & Dual-Branch Ensemble Inference (Run Everything)
Executes Stage 1 (Visual Branch optimization only) followed by Stage 2 (Joint Dual-Branch optimization with $\alpha = 1.0$, $\text{lr} = 10^{-4}$, backbone $\text{lr} = 10^{-5}$) and Ensemble Inference.   

In [27]:
from torchvision.ops import batched_nms

# 1. Prepare Demo Dataset & 5-Shot Split
# img_dir, ann_path, class_names = create_demo_neudet_coco()
ckpt = "IDEA-Research/grounding-dino-base"
processor = AutoProcessor.from_pretrained(ckpt)

sup_recs, qry_recs, class_names = build_k_shot_splits(ann_path, k_shot=5)
sup_loader = DataLoader(CDFSODDataset(img_dir, sup_recs, processor, class_names), batch_size=1, collate_fn=collate_fn)
qry_loader = DataLoader(CDFSODDataset(img_dir, qry_recs, processor, class_names), batch_size=1, shuffle=True, collate_fn=collate_fn)

# 2. Initialize Dual-Branch LMP Model on T4 GPU
model = HF_LMPGroundingDINO(ckpt, num_classes=len(class_names), num_negatives=3).cuda()
criterion = LMPSetCriterion().cuda()

# Separate Learning Rates: 1e-5 for Swin-B backbone, 1e-4 for everything else
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad], "lr": 1e-4},
    {"params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad], "lr": 1e-5},
], weight_decay=1e-4)

@torch.no_grad()
def update_support_prototypes_t4(model, sup_loader):
    model.eval()
    feats_list, boxes_list, labels_list, shapes_list = [], [], [], []
    for batch in sup_loader:
        with torch.amp.autocast("cuda", dtype=torch.float16):
            srcs, _, _ = model.extract_multi_scale_features(
                batch["pixel_values"].cuda(), batch["pixel_mask"].cuda()
            )
        feats_list.append(srcs[0].float())
        boxes_list.append([b.cuda() for b in batch["boxes_xyxy"]])
        labels_list.extend([l.cuda() for l in batch["labels"]])
        shapes_list.append(batch["img_shapes"])
    model.prototype_constructor.update_class_prototypes(feats_list, boxes_list, labels_list, shapes_list)

scaler = torch.amp.GradScaler("cuda")

# 3. Two-Stage Training Schedule (Section 4)
for stage, epochs in [(1, 2), (2, 2)]:
    print(f"\n--- Starting Stage {stage} ({'Visual Branch Only' if stage==1 else 'Joint Dual-Branch'}) ---")
    for epoch in range(epochs):
        update_support_prototypes_t4(model, sup_loader)
        model.train()
        total_loss = 0.0

        for step, batch in enumerate(qry_loader):
            optimizer.zero_grad()
            targets = [{k: v.cuda() for k, v in t.items()} for t in batch["targets"]]

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(
                    pixel_values=batch["pixel_values"].cuda(),
                    pixel_mask=batch["pixel_mask"].cuda(),
                    class_names=class_names,
                    tokenizer=processor.tokenizer,
                    gt_boxes_xyxy=[b.cuda() for b in batch["boxes_xyxy"]],
                    img_shapes=batch["img_shapes"],
                    stage=stage,
                )

                # Diagnostic
                def check_tensor(name, x):
                    x = x.detach().float()
                    finite = torch.isfinite(x)

                    print(f"\n{name}:")
                    print("  Shape:", tuple(x.shape))
                    print("  Finite:", finite.all().item())
                    print("  NaN count:", torch.isnan(x).sum().item())
                    print("  Inf count:", torch.isinf(x).sum().item())

                    if finite.any():
                        valid = x[finite]
                        print("  Min:", valid.min().item())
                        print("  Max:", valid.max().item())

                vis = outputs["visual"]

                check_tensor("Visual pred_logits", vis["pred_logits"])
                check_tensor("Visual pred_boxes", vis["pred_boxes"])

                for i, target in enumerate(targets):
                    print(f"\nTarget {i}:")
                    check_tensor("Target labels", target["labels"])
                    check_tensor("Target boxes", target["boxes"])

                    print("  Label range:",
                          target["labels"].min().item() if target["labels"].numel() else "empty",
                          target["labels"].max().item() if target["labels"].numel() else "empty")

                L_vis = criterion(outputs["visual"], targets)
                if stage == 1:
                    loss = L_vis
                else:
                    L_txt = criterion(outputs["text"], targets)
                    loss = L_txt + 1.0 * L_vis  # alpha = 1.0

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        vram_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"Stage {stage} | Epoch {epoch+1}/{epochs} | Avg Loss: {total_loss/len(qry_loader):.4f} | Peak VRAM: {vram_gb:.2f} GB")

# 4. Dual-Branch Ensemble Inference (Section 3.5)
@torch.no_grad()
def predict_ensemble(model, batch, class_names, tokenizer, top_k=100, nms_iou=0.5):
    model.eval()
    with torch.amp.autocast("cuda", dtype=torch.float16):
        out = model(
            pixel_values=batch["pixel_values"].cuda(),
            pixel_mask=batch["pixel_mask"].cuda(),
            class_names=class_names,
            tokenizer=tokenizer,
            stage=2,
        )
    # Pool 900 text queries + 900 visual queries -> 1800 candidate predictions
    probs = torch.cat([out["text"]["pred_logits"].sigmoid(), out["visual"]["pred_logits"].sigmoid()], dim=1)[0]
    boxes = torch.cat([out["text"]["pred_boxes"], out["visual"]["pred_boxes"]], dim=1)[0]

    H, W = batch["img_shapes"][0]
    scores, idx = torch.topk(probs.view(-1), k=min(top_k, probs.numel()))
    box_idx, labels = idx // len(class_names), idx % len(class_names)

    boxes_xyxy = box_convert(boxes[box_idx].float(), "cxcywh", "xyxy")
    boxes_xyxy *= torch.tensor([W, H, W, H], device=boxes.device)
    keep = batched_nms(boxes_xyxy, scores, labels, iou_threshold=nms_iou)
    return boxes_xyxy[keep], scores[keep], labels[keep]

sample_batch = next(iter(qry_loader))
pred_boxes, pred_scores, pred_labels = predict_ensemble(model, sample_batch, class_names, processor.tokenizer)
print(f"\nEnsemble Inference Complete: Top Detection -> Class '{class_names[pred_labels[0].item()]}' (Score: {pred_scores[0].item():.3f})")

Loading weights:   0%|          | 0/1182 [00:00<?, ?it/s]


--- Starting Stage 1 (Visual Branch Only) ---

Visual pred_logits:
  Shape: (1, 900, 6)
  Finite: False
  NaN count: 5400
  Inf count: 0

Visual pred_boxes:
  Shape: (1, 900, 4)
  Finite: False
  NaN count: 3600
  Inf count: 0

Target 0:

Target labels:
  Shape: (1,)
  Finite: True
  NaN count: 0
  Inf count: 0
  Min: 5.0
  Max: 5.0

Target boxes:
  Shape: (1, 4)
  Finite: True
  NaN count: 0
  Inf count: 0
  Min: 0.1599999964237213
  Max: 0.9900000095367432
  Label range: 5 5


ValueError: matrix contains invalid numeric entries

# Diagnostics

In [ ]:
# Inspect actual Hugging Face backbone outputs
model.eval()

batch = next(iter(sup_loader))

with torch.no_grad():
    features, positions = model.backbone(
        batch["pixel_values"].cuda(),
        batch["pixel_mask"].cuda()
    )

print("Number of feature levels:", len(features))

for i, (feature, mask) in enumerate(features):
    print(
        f"Level {i}: "
        f"Shape={tuple(feature.shape)}, "
        f"Channels={feature.shape[1]}, "
        f"Mask={tuple(mask.shape)}"
    )

print("\nProjection layers:")
for i, layer in enumerate(model.input_proj):
    print(f"Level {i}: {layer[0].in_channels} -> {layer[0].out_channels}")

In [ ]:
model = HF_LMPGroundingDINO(
    ckpt,
    num_classes=len(class_names),
    num_negatives=3
).cuda()

model.eval()

batch = next(iter(sup_loader))

with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    srcs, masks, pos_embeds = model.extract_multi_scale_features(
        batch["pixel_values"].cuda(),
        batch["pixel_mask"].cuda()
    )

print("Feature pyramid verification:")

for i in range(len(srcs)):
    print(
        f"Level {i}: "
        f"Features={tuple(srcs[i].shape)}, "
        f"Mask={tuple(masks[i].shape)}, "
        f"Position={tuple(pos_embeds[i].shape)}"
    )

assert len(srcs) == 4, "Expected exactly 4 feature levels"

assert all(x.shape[1] == 256 for x in srcs), \
    "All feature levels must have 256 channels"

assert len(masks) == len(pos_embeds) == 4

print("\nFeature pyramid verification PASSED!")